In [9]:
# ============================================================
# DAY 8 — FULL UNIVERSE EXPANSION
# AI Stock Screener — Indian Markets
# Stage 1: yfinance pre-filters on full 4,959 stock universe
# ============================================================

# ── CELL 1: IMPORTS & SETUP ──────────────────────────────────
import pandas as pd
import numpy as np
import yfinance as yf
import pickle
import os
import time
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data', exist_ok=True)

print("✅ Imports done")
print(f"yfinance version: {yf.__version__}")

✅ Imports done
yfinance version: 1.2.0


In [10]:
# ── CELL 2: LOAD MASTER STOCKS ───────────────────────────────
master = pd.read_csv('../data/master_stocks.csv')

print(f"Total stocks in universe : {len(master)}")
print(f"NSE stocks               : {len(master[master['Exchange'] == 'NSE'])}")
print(f"BSE stocks               : {len(master[master['Exchange'] == 'BSE'])}")
print(f"\nSample:")
print(master.head(5).to_string(index=False))

Total stocks in universe : 4959
NSE stocks               : 2117
BSE stocks               : 2842

Sample:
    Symbol                             Company Name Exchange
 20MICRONS                       20 Microns Limited      NSE
21STCENMGM 21st Century Management Services Limited      NSE
    360ONE                      360 ONE WAM LIMITED      NSE
 3IINFOLTD                      3i Infotech Limited      NSE
   3MINDIA                         3M India Limited      NSE


In [11]:
# ── CELL 3: REMOVE COMPANY NAME DUPLICATES ───────────────────
# Symbol dedup already done in Day 1
# But same company can appear as NSE+BSE with different symbols
# Strategy: if company name matches (fuzzy), keep NSE, drop BSE

nse_stocks = master[master['Exchange'] == 'NSE'].copy()
bse_stocks = master[master['Exchange'] == 'BSE'].copy()

# Normalize company names for matching
def normalize_name(name):
    name = str(name).upper().strip()
    # Remove common suffixes
    for suffix in [' LIMITED', ' LTD', ' LTD.', ' PRIVATE', 
                   ' PVT', ' PVT.', ' INC', ' CORP', ' CO.']:
        name = name.replace(suffix, '')
    # Remove special characters
    name = ''.join(e for e in name if e.isalnum() or e == ' ')
    return name.strip()

nse_stocks['Name_Norm'] = nse_stocks['Company Name'].apply(normalize_name)
bse_stocks['Name_Norm'] = bse_stocks['Company Name'].apply(normalize_name)

# Find BSE stocks whose normalized name exists in NSE
nse_names = set(nse_stocks['Name_Norm'])
bse_duplicates = bse_stocks[bse_stocks['Name_Norm'].isin(nse_names)]
bse_unique     = bse_stocks[~bse_stocks['Name_Norm'].isin(nse_names)]

print(f"NSE stocks              : {len(nse_stocks)}")
print(f"BSE stocks (original)   : {len(bse_stocks)}")
print(f"BSE duplicates (by name): {len(bse_duplicates)}")
print(f"BSE unique (keep)       : {len(bse_unique)}")

# Final clean universe
master_clean = pd.concat([nse_stocks, bse_unique], ignore_index=True)
master_clean = master_clean.drop(columns=['Name_Norm'])

print(f"\nFinal clean universe    : {len(master_clean)}")
print(f"  NSE : {len(master_clean[master_clean['Exchange'] == 'NSE'])}")
print(f"  BSE : {len(master_clean[master_clean['Exchange'] == 'BSE'])}")

# Show sample duplicates caught
print(f"\nSample BSE duplicates removed:")
print(bse_duplicates[['Symbol', 'Company Name']].head(10).to_string(index=False))

NSE stocks              : 2117
BSE stocks (original)   : 2842
BSE duplicates (by name): 6
BSE unique (keep)       : 2836

Final clean universe    : 4953
  NSE : 2117
  BSE : 2836

Sample BSE duplicates removed:
   Symbol                          Company Name
   SINGER                  Singer India Limited
     GVTD          GE Vernova T&D India Limited
SOLARAPP1 SOLARA ACTIVE PHARMA SCIENCES LIMITED
   SSFLPP   Spandana Sphoorty Financial Limited
    ATLPP            Allcargo Terminals Limited
  KRISHPP               KRISHIVAL FOODS LIMITED


In [12]:
# ── CELL 4: BUILD YFINANCE TICKER SYMBOLS ────────────────────

def get_ticker(symbol, exchange):
    if exchange == 'NSE':
        return f"{symbol}.NS"
    else:
        return f"{symbol}.BO"

master_clean['Ticker'] = master_clean.apply(
    lambda row: get_ticker(row['Symbol'], row['Exchange']), axis=1
)

print(f"Total tickers: {len(master_clean)}")
print(f"\nSample NSE tickers:")
print(master_clean[master_clean['Exchange']=='NSE']['Ticker'].head(5).tolist())
print(f"\nSample BSE tickers:")
print(master_clean[master_clean['Exchange']=='BSE']['Ticker'].head(5).tolist())

Total tickers: 4953

Sample NSE tickers:
['20MICRONS.NS', '21STCENMGM.NS', '360ONE.NS', '3IINFOLTD.NS', '3MINDIA.NS']

Sample BSE tickers:
['AMBALALSA.BO', 'ANDHRAPET.BO', 'ANSALAPI.BO', 'UTIQUE.BO', 'ARUNAHTEL.BO']


In [16]:
# ── CELL 5: PRE-FILTER ─────────────────────

from datetime import datetime

MIN_DAILY_VALUE  = 10_00_000
MIN_MARKET_CAP   = 50_00_00_000
MIN_PRICE        = 5
MIN_LISTING_DAYS = 365
MIN_DATA_ROWS    = 200

def flatten_df(df):
    """Flatten MultiIndex columns from yfinance 1.2.0"""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]
    return df

def check_stock(symbol, exchange, ticker):
    result = {
        'Symbol'           : symbol,
        'Exchange'         : exchange,
        'Ticker'           : ticker,
        'passed'           : False,
        'fail_reason'      : None,
        'Price'            : None,
        'Avg_Daily_Value_L': None,
        'Market_Cap_Cr'    : None,
        'Data_Rows'        : None,
    }
    try:
        df_60 = yf.download(ticker, period='60d', interval='1d',
                            progress=False, auto_adjust=True)
        df_60 = flatten_df(df_60)

        if df_60 is None or len(df_60) < 5:
            result['fail_reason'] = 'No data'
            return result

        df_full = yf.download(ticker, period='max', interval='1d',
                              progress=False, auto_adjust=True)
        df_full = flatten_df(df_full)

        current_price = float(df_60['Close'].iloc[-1])
        avg_value     = float(df_60['Volume'].mean()) * current_price

        result['Price']             = round(current_price, 2)
        result['Avg_Daily_Value_L'] = round(avg_value / 1e5, 2)
        result['Data_Rows']         = len(df_full)

        # Filter checks
        if current_price < MIN_PRICE:
            result['fail_reason'] = f'Price Rs {current_price:.1f} < Rs 5'
            return result

        if avg_value < MIN_DAILY_VALUE:
            result['fail_reason'] = f'Daily value Rs {avg_value/1e5:.1f}L < Rs 10L'
            return result

        market_cap = None
        try:
            info       = yf.Ticker(ticker).fast_info
            market_cap = getattr(info, 'market_cap', None)
        except:
            pass

        if market_cap is not None:
            result['Market_Cap_Cr'] = round(market_cap / 1e7, 2)
            if market_cap < MIN_MARKET_CAP:
                result['fail_reason'] = f'MCap Rs {market_cap/1e7:.1f}Cr < Rs 50Cr'
                return result

        if len(df_full) > 0:
            first_date   = df_full.index[0]
            listing_days = (datetime.now() - first_date.to_pydatetime().replace(tzinfo=None)).days
            if listing_days < MIN_LISTING_DAYS:
                result['fail_reason'] = f'Listed only {listing_days} days'
                return result

        if len(df_full) < MIN_DATA_ROWS:
            result['fail_reason'] = f'Only {len(df_full)} rows'
            return result

        result['passed'] = True
        return result

    except Exception as e:
        result['fail_reason'] = f'Error: {str(e)[:60]}'
        return result

print("✅ Filter definitions ready")

✅ Filter definitions ready


In [18]:
# ── CELL 6: FULL UNIVERSE PRE-FILTER ─────────────────────────
# Set to True ONLY when you want to force a fresh quarterly run
FORCE_RERUN = False

if not FORCE_RERUN and os.path.exists('../data/prefilt_passed.csv'):
    existing = pd.read_csv('../data/prefilt_passed.csv')
    print(f"⚠️  Pre-filter already completed — {len(existing)} stocks")
    print(f"    Set FORCE_RERUN = True to re-run quarterly refresh")

else:
    CHECKPOINT_FILE = '../data/prefilt_checkpoint.pkl'
    OUTPUT_FILE     = '../data/prefilt_results.csv'

    def is_checkpoint_valid(checkpoint):
        results = checkpoint.get('results', [])
        if len(results) < 50:
            return True
        first_50      = results[:50]
        first_50_pass = sum(1 for r in first_50 if r.get('passed', False))
        pass_rate     = first_50_pass / 50
        if pass_rate == 0:
            print(f"  ⚠️  Checkpoint invalid — 0% pass rate, starting fresh")
            return False
        has_prices = any(r.get('Price') is not None for r in first_50)
        if not has_prices:
            print(f"  ⚠️  Checkpoint invalid — no price data, starting fresh")
            return False
        print(f"  ✅ Checkpoint valid — first 50 pass rate: {pass_rate:.0%}")
        return True

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            checkpoint = pickle.load(f)
        if is_checkpoint_valid(checkpoint):
            results   = checkpoint['results']
            start_idx = checkpoint['next_idx']
            print(f"Resuming from stock {start_idx}/{len(master_clean)}")
        else:
            os.remove(CHECKPOINT_FILE)
            results   = []
            start_idx = 0
            print(f"Starting fresh — {len(master_clean)} stocks")
    else:
        results   = []
        start_idx = 0
        print(f"Starting fresh — {len(master_clean)} stocks")

    total  = len(master_clean)
    passed = sum(1 for r in results if r.get('passed', False))
    failed = len(results) - passed

    print(f"Already processed: {len(results)} | Passed: {passed} | Failed: {failed}")
    print("-" * 60)

    for i in range(start_idx, total):
        row    = master_clean.iloc[i]
        symbol = row['Symbol']
        exch   = row['Exchange']
        ticker = row['Ticker']

        r = check_stock(symbol, exch, ticker)
        results.append(r)

        if r['passed']:
            passed += 1
        else:
            failed += 1

        if (i + 1) % 50 == 0 or (i + 1) == total:
            pct = (i + 1) / total * 100
            print(f"  [{i+1:4d}/{total}] {pct:5.1f}% | Passed: {passed} | Failed: {failed}")

        if (i + 1) % 100 == 0:
            with open(CHECKPOINT_FILE, 'wb') as f:
                pickle.dump({
                    'results'   : results,
                    'next_idx'  : i + 1,
                    'version'   : '2.0',
                    'created_at': datetime.now().strftime('%Y-%m-%d %H:%M'),
                }, f)

        time.sleep(0.5)

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_FILE, index=False)

    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

    passed_df = results_df[results_df['passed'] == True]

    print(f"\n✅ Pre-filter complete")
    print(f"   Total processed : {len(results_df)}")
    print(f"   Passed          : {len(passed_df)}")
    print(f"   Failed          : {len(results_df) - len(passed_df)}")
    print(f"\nTop fail reasons:")
    print(results_df[results_df['passed'] == False]['fail_reason']
          .str.split('<').str[0].str.strip()
          .value_counts().head(10).to_string())

    passed_df.to_csv('../data/prefilt_passed.csv', index=False)
    print(f"\n✅ Passed stocks saved: data/prefilt_passed.csv")

⚠️  Pre-filter already completed — 2142 stocks
    Set FORCE_RERUN = True to re-run quarterly refresh


In [19]:
# ── CELL 7: ANALYSE PRE-FILTER RESULTS ───────────────────────
passed_df = pd.read_csv('../data/prefilt_passed.csv')

print(f"Total passed : {len(passed_df)}")
print(f"\nExchange breakdown:")
print(passed_df['Exchange'].value_counts().to_string())

print(f"\nMarket cap distribution:")
passed_df['Cap_Category'] = pd.cut(
    passed_df['Market_Cap_Cr'],
    bins=[0, 1000, 5000, 20000, float('inf')],
    labels=['Small Cap', 'Mid Cap', 'Mini Large', 'Large Cap']
)
print(passed_df['Cap_Category'].value_counts().to_string())

print(f"\nDaily value distribution (Lakhs):")
bins   = [0, 10, 50, 100, 500, 1000, float('inf')]
labels = ['<10L', '10-50L', '50-100L', '100-500L', '500L-1Cr', '>1Cr']
passed_df['Val_Bucket'] = pd.cut(passed_df['Avg_Daily_Value_L'], bins=bins, labels=labels)
print(passed_df['Val_Bucket'].value_counts().sort_index().to_string())

print(f"\nSample passed stocks:")
print(passed_df[['Symbol', 'Exchange', 'Price', 'Avg_Daily_Value_L', 'Market_Cap_Cr']].head(10).to_string(index=False))

Total passed : 2142

Exchange breakdown:
Exchange
NSE    1692
BSE     450

Market cap distribution:
Cap_Category
Small Cap     881
Mid Cap       578
Mini Large    366
Large Cap     306

Daily value distribution (Lakhs):
Val_Bucket
<10L          0
10-50L      591
50-100L     186
100-500L    435
500L-1Cr    206
>1Cr        724

Sample passed stocks:
    Symbol Exchange    Price  Avg_Daily_Value_L  Market_Cap_Cr
 20MICRONS      NSE   151.02             495.99         534.54
    360ONE      NSE  1040.60           10101.78       42239.61
 3IINFOLTD      NSE    13.82              62.49         287.09
   3MINDIA      NSE 32270.00            1477.32       36352.38
    5PAISA      NSE   291.60             627.78         911.30
   63MOONS      NSE   531.45             916.80        2448.99
   AAATECH      NSE    93.00              49.10         119.29
 AADHARHFC      NSE   449.85            1868.57       19561.12
AAREYDRUGS      NSE    69.28             169.96         196.47
     AARON      NSE 

In [24]:
passed_df = pd.read_csv('../data/prefilt_passed.csv')
print(f"Stocks to scrape: {len(passed_df)}")

Stocks to scrape: 2142


In [38]:
# ── CELL 8: FULL UNIVERSE FUNDAMENTAL SCRAPER ────────────────
# Handles NSE symbols + BSE numeric code fallback automatically
# Checkpointing, resume, FORCE_RESCRAPE guard

from bs4 import BeautifulSoup
import requests

FUND_CHECKPOINT = '../data/fund_scrape_checkpoint.pkl'
FUND_RAW_FILE   = '../data/raw_stock_data_full.pkl'
FUND_OUTPUT     = '../data/fundamental_metrics_full.csv'
FORCE_RESCRAPE  = False

# ── GUARD ────────────────────────────────────────────────────
if not FORCE_RESCRAPE and os.path.exists(FUND_OUTPUT):
    existing = pd.read_csv(FUND_OUTPUT)
    print(f"⚠️  Fundamental scrape already completed — {len(existing)} stocks")
    print(f"    Set FORCE_RESCRAPE = True to re-run quarterly refresh")

else:
    # ── BSE CODE MAP ─────────────────────────────────────────
    bse_raw = pd.read_csv('../data/bse_stocks.csv',
                          index_col=False,
                          usecols=['Security Code', 'Security Id',
                                   'Issuer Name', 'Status',
                                   'Group', 'Face Value',
                                   'ISIN No', 'Instrument'])
    bse_raw['Security Id'] = bse_raw['Security Id'].str.strip()
    bse_code_map = dict(zip(bse_raw['Security Id'],
                            bse_raw['Security Code']))
    print(f"✅ BSE code map loaded: {len(bse_code_map)} symbols")

    # ── SCRAPER FUNCTIONS ────────────────────────────────────
    def extract_table(table):
        try:
            thead   = table.find('thead')
            columns = []
            if thead:
                for th in thead.find_all('th'):
                    text = th.get_text(strip=True)
                    columns.append(text if text else 'Metric')
            tbody = table.find('tbody')
            if not tbody:
                return None
            rows = tbody.find_all('tr')
            data = {}
            for row in rows:
                cells = row.find_all('td')
                if not cells:
                    continue
                metric_name = cells[0].get_text(strip=True)
                metric_name = metric_name.replace('+', '').strip()
                skip_keywords = ['Raw PDF', 'PDF', 'Source']
                if any(kw in metric_name for kw in skip_keywords):
                    continue
                values = []
                for cell in cells[1:]:
                    val = cell.get_text(strip=True)
                    val = val.replace(',', '').replace('%', '').strip()
                    try:
                        values.append(float(val))
                    except:
                        values.append(val if val else None)
                data[metric_name] = values
            if not data:
                return None
            col_names = columns[1:] if len(columns) > 1 else list(range(len(values)))
            return pd.DataFrame(data, index=col_names).T
        except:
            return None

    def scrape_screener(lookup):
        """
        Scrape Screener.in for a given lookup (NSE symbol or BSE code).
        Tries consolidated first, falls back to standalone.
        """
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
            urls = [
                f"https://www.screener.in/company/{lookup}/consolidated/",
                f"https://www.screener.in/company/{lookup}/",
            ]
            soup = None
            for url in urls:
                response = requests.get(url, headers=headers, timeout=15)
                if response.status_code != 200:
                    continue
                temp_soup = BeautifulSoup(response.text, 'html.parser')
                # Check if P&L has real data
                has_data = False
                for section in temp_soup.find_all('section'):
                    heading = section.find('h2')
                    if heading and 'Profit' in heading.text:
                        table = section.find('table')
                        if table:
                            thead = table.find('thead')
                            if thead and len(thead.find_all('th')) > 1:
                                has_data = True
                        break
                if has_data:
                    soup = temp_soup
                    break
            if soup is None:
                return None

            target_sections = [
                'Quarterly Results', 'Profit & Loss', 'Balance Sheet',
                'Cash Flows', 'Ratios', 'Shareholding Pattern',
            ]
            result = {}
            for section in soup.find_all('section'):
                heading = section.find('h2')
                if not heading:
                    continue
                section_name = heading.text.strip()
                if section_name not in target_sections:
                    continue
                table = section.find('table')
                if not table:
                    continue
                df = extract_table(table)
                if df is not None:
                    result[section_name] = df
            for section in soup.find_all('section'):
                heading = section.find('h2')
                if heading and 'Peer' in heading.text:
                    links       = section.find_all('a')
                    sector_info = {}
                    titles      = ['Broad Sector', 'Sector',
                                   'Broad Industry', 'Industry']
                    for link in links:
                        title = link.get('title', '')
                        if title in titles:
                            sector_info[title] = link.text.strip()
                    result['Sector Info'] = sector_info
                    break
            return result if result else None
        except:
            return None

    def scrape_stock(symbol, exchange):
        """
        Smart scraper — tries NSE symbol first.
        If fails and BSE stock, retries with BSE numeric code.
        """
        # Try NSE symbol first (works for all NSE + many BSE)
        result = scrape_screener(symbol)
        if result:
            return result

        # If BSE stock and symbol lookup failed → try BSE numeric code
        if exchange == 'BSE':
            bse_code = bse_code_map.get(symbol)
            if bse_code:
                result = scrape_screener(str(int(bse_code)))
                if result:
                    return result

        return None

    def safe_cagr(end_val, start_val, years):
        try:
            if end_val and start_val and start_val > 0 and end_val > 0 and years > 0:
                return round(((end_val / start_val) ** (1/years) - 1) * 100, 2)
        except:
            pass
        return None

    def extract_all_metrics(symbol, data):
        m = {'Symbol': symbol}
        try:
            sector_info       = data.get('Sector Info', {})
            m['Sector']       = sector_info.get('Sector', None)
            m['Industry']     = sector_info.get('Industry', None)
            m['Broad Sector'] = sector_info.get('Broad Sector', None)

            financial_keywords = ['Financial', 'Banking', 'Insurance',
                                   'NBFC', 'Lending', 'Microfinance']
            is_financial = any(kw in str(m.get('Sector', ''))
                               for kw in financial_keywords)

            if is_financial:
                revenue_row = 'Revenue'
                margin_row  = 'Financing Margin %'
                debt_row    = 'Borrowing'
            else:
                revenue_row = 'Sales'
                margin_row  = 'OPM %'
                debt_row    = 'Borrowings'

            pl = data.get('Profit & Loss')
            if pl is not None:
                pl_clean = pl.drop(columns=['TTM'], errors='ignore')
                if revenue_row in pl_clean.index:
                    sales = pl_clean.loc[revenue_row].dropna()
                    if len(sales) >= 6:
                        m['Revenue CAGR 5Y'] = safe_cagr(
                            sales.iloc[-1], sales.iloc[-6], 5)
                    if len(sales) >= 11:
                        m['Revenue CAGR 10Y'] = safe_cagr(
                            sales.iloc[-1], sales.iloc[-11], 10)
                    if len(sales) >= 3:
                        yrs = min(len(sales) - 1, 10)
                        base = sales.iloc[-yrs - 1]
                        if base > 0:
                            m['Revenue CAGR Max']  = safe_cagr(
                                sales.iloc[-1], base, yrs)
                            m['Revenue CAGR Years'] = yrs

                if margin_row in pl_clean.index:
                    opm = pl_clean.loc[margin_row].dropna()
                    m['Avg OPM 5Y']  = round(opm.iloc[-5:].mean(), 2) if len(opm) >= 5 else (round(opm.mean(), 2) if len(opm) > 0 else None)
                    m['Avg OPM 10Y'] = round(opm.iloc[-10:].mean(), 2) if len(opm) >= 10 else None

                if 'Net Profit' in pl_clean.index:
                    profit = pl_clean.loc['Net Profit'].dropna()
                    if len(profit) >= 6 and profit.iloc[-6] > 0:
                        m['Profit CAGR 5Y'] = safe_cagr(
                            profit.iloc[-1], profit.iloc[-6], 5)
                    if len(profit) >= 11 and profit.iloc[-11] > 0:
                        m['Profit CAGR 10Y'] = safe_cagr(
                            profit.iloc[-1], profit.iloc[-11], 10)
                    if len(profit) >= 3:
                        yrs = min(len(profit) - 1, 10)
                        base = profit.iloc[-yrs - 1]
                        if base > 0:
                            m['Profit CAGR Max']  = safe_cagr(
                                profit.iloc[-1], base, yrs)
                            m['Profit CAGR Years'] = yrs

            qr = data.get('Quarterly Results')
            if qr is not None:
                qr_clean = qr.drop(columns=['TTM'], errors='ignore')
                if revenue_row in qr_clean.index:
                    sales_q = qr_clean.loc[revenue_row].dropna()
                    if len(sales_q) >= 4:
                        m['TTM Revenue'] = round(sales_q.iloc[-4:].sum(), 2)
                    if len(sales_q) >= 5:
                        yr_ago = sales_q.iloc[-5]
                        if yr_ago > 0:
                            m['Revenue YoY Q'] = round(
                                (sales_q.iloc[-1] - yr_ago) / yr_ago * 100, 2)
                    yoy = 0
                    for i in range(1, min(5, len(sales_q) - 4)):
                        curr = sales_q.iloc[-i]
                        prev = sales_q.iloc[-i - 4]
                        if prev > 0 and curr > prev:
                            yoy += 1
                        else:
                            break
                    m['Revenue Consecutive YoY'] = yoy
                    if len(sales_q) >= 9:
                        r1 = (sales_q.iloc[-1] - sales_q.iloc[-5]) / sales_q.iloc[-5] * 100
                        r2 = (sales_q.iloc[-5] - sales_q.iloc[-9]) / sales_q.iloc[-9] * 100
                        m['Revenue Accelerating'] = r1 > r2

                if 'Net Profit' in qr_clean.index:
                    profit_q = qr_clean.loc['Net Profit'].dropna()
                    if len(profit_q) >= 4:
                        m['TTM Profit'] = round(profit_q.iloc[-4:].sum(), 2)
                    if len(profit_q) >= 5:
                        yr_ago = profit_q.iloc[-5]
                        if yr_ago > 0:
                            m['Profit YoY Q'] = round(
                                (profit_q.iloc[-1] - yr_ago) / yr_ago * 100, 2)
                    m['Profit Positive Q'] = int((profit_q > 0).sum())
                    m['Total Quarters']    = len(profit_q)

                if margin_row in qr_clean.index:
                    opm_q = qr_clean.loc[margin_row].dropna()
                    m['Latest OPM Q'] = opm_q.iloc[-1] if len(opm_q) > 0 else None
                    m['Avg OPM 4Q']   = round(opm_q.iloc[-4:].mean(), 2) if len(opm_q) >= 4 else None
                    if len(opm_q) >= 5:
                        m['Margin Improving'] = bool(opm_q.iloc[-1] >= opm_q.iloc[-5])

                if 'EPS in Rs' in qr_clean.index:
                    eps_q = qr_clean.loc['EPS in Rs'].dropna()
                    m['Latest EPS Q'] = eps_q.iloc[-1] if len(eps_q) > 0 else None
                    if len(eps_q) >= 5 and eps_q.iloc[-5] > 0:
                        m['EPS YoY Growth'] = round(
                            (eps_q.iloc[-1] - eps_q.iloc[-5]) / eps_q.iloc[-5] * 100, 2)

            bs = data.get('Balance Sheet')
            if bs is not None:
                if debt_row in bs.index:
                    debt = bs.loc[debt_row].dropna()
                    m['Latest Debt']   = debt.iloc[-1] if len(debt) > 0 else None
                    if len(debt) >= 4:
                        m['Debt Reducing'] = bool(debt.iloc[-1] < debt.iloc[-4])
                if 'Reserves' in bs.index and 'Equity Capital' in bs.index:
                    reserves = bs.loc['Reserves'].dropna()
                    eq_cap   = bs.loc['Equity Capital'].dropna()
                    if len(reserves) > 0 and len(eq_cap) > 0:
                        equity           = reserves.iloc[-1] + eq_cap.iloc[-1]
                        m['Latest Equity'] = round(equity, 2)
                        if m.get('TTM Profit') and equity > 0:
                            m['ROE'] = round(m['TTM Profit'] / equity * 100, 2)
                        if m.get('Latest Debt') is not None and equity > 0:
                            m['Debt to Equity'] = round(m['Latest Debt'] / equity, 2)

            cf = data.get('Cash Flows')
            if cf is not None:
                if 'Cash from Operating Activity' in cf.index:
                    op_cf = cf.loc['Cash from Operating Activity'].dropna()
                    m['Latest Operating CF'] = op_cf.iloc[-1] if len(op_cf) > 0 else None
                    m['CF Positive Years']   = int((op_cf > 0).sum())
                    m['CF Total Years']      = len(op_cf)
                    if len(op_cf) >= 4:
                        m['CF Growing'] = bool(op_cf.iloc[-1] > op_cf.iloc[-4])

            ratios = data.get('Ratios')
            if ratios is not None:
                if 'ROCE %' in ratios.index:
                    roce = ratios.loc['ROCE %'].dropna()
                    m['Latest ROCE']  = roce.iloc[-1] if len(roce) > 0 else None
                    if len(roce) >= 4:
                        m['ROCE Improving'] = bool(roce.iloc[-1] > roce.iloc[-4])
                    if len(roce) >= 5:
                        m['Avg ROCE 5Y'] = round(roce.iloc[-5:].mean(), 2)
                if 'ROE %' in ratios.index:
                    roe_s = ratios.loc['ROE %'].dropna()
                    m['ROE from Ratios'] = roe_s.iloc[-1] if len(roe_s) > 0 else None
                    m['ROE Avg 5Y']      = round(roe_s.iloc[-5:].mean(), 2) if len(roe_s) >= 5 else None
                    m['ROE Improving']   = bool(roe_s.iloc[-1] > roe_s.iloc[-4]) if len(roe_s) >= 4 else None

            # Final ROE
            if pd.notna(m.get('ROE from Ratios')):
                m['Final ROE'] = m['ROE from Ratios']
            elif pd.notna(m.get('ROE')):
                m['Final ROE'] = m['ROE']
            else:
                m['Final ROE'] = None

            sh = data.get('Shareholding Pattern')
            if sh is not None:
                if 'Promoters' in sh.index:
                    promoter = sh.loc['Promoters'].dropna()
                    m['Promoter Holding']   = promoter.iloc[-1] if len(promoter) > 0 else None
                    if len(promoter) >= 5:
                        m['Promoter Change 4Q'] = round(promoter.iloc[-1] - promoter.iloc[-5], 2)
                    if len(promoter) >= 9:
                        m['Promoter Change 8Q'] = round(promoter.iloc[-1] - promoter.iloc[-9], 2)
                if 'FIIs' in sh.index:
                    fii = sh.loc['FIIs'].dropna()
                    m['FII Holding']   = fii.iloc[-1] if len(fii) > 0 else None
                    if len(fii) >= 5:
                        m['FII Change 4Q'] = round(fii.iloc[-1] - fii.iloc[-5], 2)
                if 'DIIs' in sh.index:
                    dii = sh.loc['DIIs'].dropna()
                    m['DII Holding']   = dii.iloc[-1] if len(dii) > 0 else None
                    if len(dii) >= 5:
                        m['DII Change 4Q'] = round(dii.iloc[-1] - dii.iloc[-5], 2)

        except Exception as e:
            pass
        return m

    # ── LOAD CHECKPOINT ──────────────────────────────────────
    if os.path.exists(FUND_CHECKPOINT):
        with open(FUND_CHECKPOINT, 'rb') as f:
            ckpt = pickle.load(f)
        all_stock_data = ckpt['raw_data']
        all_metrics    = ckpt['metrics']
        start_idx      = ckpt['next_idx']
        print(f"Resuming from stock {start_idx}/{len(passed_df)}")
    else:
        all_stock_data = {}
        all_metrics    = []
        start_idx      = 0
        print(f"Starting fresh — {len(passed_df)} stocks to scrape")

    failed_stocks = []
    total         = len(passed_df)

    print(f"Already scraped: {len(all_metrics)} stocks")
    print(f"Estimated time : {(total - start_idx) * 3 / 60:.0f} minutes")
    print("-" * 60)

    for i in range(start_idx, total):
        row      = passed_df.iloc[i]
        symbol   = row['Symbol']
        exchange = row['Exchange']

        try:
            # Smart scraper — NSE first, BSE code fallback
            data = scrape_stock(symbol, exchange)
            if data:
                all_stock_data[symbol] = data
                metrics = extract_all_metrics(symbol, data)
                all_metrics.append(metrics)
                sections = len([k for k in data.keys() if k != 'Sector Info'])
                status   = f"✓ ({sections} sections)"
            else:
                failed_stocks.append(symbol)
                status = "✗ No data"

        except Exception as e:
            failed_stocks.append(symbol)
            status = "✗ Error"

        if (i + 1) % 25 == 0 or (i + 1) == total:
            pct = (i + 1) / total * 100
            print(f"  [{i+1:4d}/{total}] {pct:5.1f}% | "
                  f"Scraped: {len(all_metrics)} | Failed: {len(failed_stocks)}")

        if (i + 1) % 50 == 0:
            with open(FUND_CHECKPOINT, 'wb') as f:
                pickle.dump({
                    'raw_data' : all_stock_data,
                    'metrics'  : all_metrics,
                    'next_idx' : i + 1,
                }, f)

        time.sleep(2.5)

    # ── SAVE RESULTS ─────────────────────────────────────────
    with open(FUND_RAW_FILE, 'wb') as f:
        pickle.dump(all_stock_data, f)

    metrics_df = pd.DataFrame(all_metrics)
    metrics_df.to_csv(FUND_OUTPUT, index=False)

    if os.path.exists(FUND_CHECKPOINT):
        os.remove(FUND_CHECKPOINT)

    print(f"\n✅ Fundamental scrape complete")
    print(f"   Scraped  : {len(metrics_df)} stocks")
    print(f"   Failed   : {len(failed_stocks)}")
    print(f"   Columns  : {len(metrics_df.columns)}")
    if failed_stocks:
        print(f"\n   Failed stocks: {failed_stocks[:20]}")
    print(f"\n✅ Saved: {FUND_OUTPUT}")

⚠️  Fundamental scrape already completed — 2142 stocks
    Set FORCE_RESCRAPE = True to re-run quarterly refresh


In [37]:
# ── CELL 10 (FINAL): TIERED QUALITY FILTER ───────────────────

fund_df    = pd.read_csv('../data/fundamental_metrics_full.csv')
prefilt_df = pd.read_csv('../data/prefilt_passed.csv')

fund_df = fund_df.merge(
    prefilt_df[['Symbol', 'Market_Cap_Cr']],
    on='Symbol', how='left'
)

print(f"Total stocks for quality filter: {len(fund_df)}")

def passes_tiered_filter(row):
    mcap   = row.get('Market_Cap_Cr', 0) or 0
    sector = str(row.get('Sector', ''))

    # ── Cap tier thresholds ───────────────────────────────────
    if mcap >= 20000:
        tier         = 'Large Cap'
        min_roe      = 6
        max_de       = 2.5
        min_promo    = 15
        min_rev      = 200
        min_cf_years = 2
    elif mcap >= 5000:
        tier         = 'Mini Large'
        min_roe      = 7
        max_de       = 2.0
        min_promo    = 20
        min_rev      = 100
        min_cf_years = 2
    elif mcap >= 1000:
        tier         = 'Mid Cap'
        min_roe      = 8
        max_de       = 1.5
        min_promo    = 25
        min_rev      = 50
        min_cf_years = 2
    else:
        tier         = 'Small Cap'
        min_roe      = 10
        max_de       = 1.0
        min_promo    = 30
        min_rev      = 20
        min_cf_years = 2

    # ── Company type detection ────────────────────────────────
    financial_sectors = ['Financial Services', 'Banking', 'Insurance']
    is_financial      = any(s in sector for s in financial_sectors)

    lumpy_cf_sectors  = ['Capital Goods', 'Consumer Durables', 'Consumer Services']
    is_lumpy_cf       = any(s in sector for s in lumpy_cf_sectors)

    promoter = float(row.get('Promoter Holding') or 0)
    fii      = float(row.get('FII Holding')      or 0)
    dii      = float(row.get('DII Holding')      or 0)

    is_mnc           = (promoter < 5 and (fii > 40 or dii > 40))
    is_institutional = (promoter < 20 and (fii + dii) > 65)

    # ── Rule 1: TTM Profit > 0 ────────────────────────────────
    profit = row.get('TTM Profit')
    if profit is None or pd.isna(profit) or profit <= 0:
        return False, 'TTM Profit negative or zero', tier

    # ── Rule 2: ROE ───────────────────────────────────────────
    roe = row.get('Final ROE') or row.get('ROE from Ratios') or row.get('ROE')
    if roe is None or pd.isna(roe) or roe < min_roe:
        return False, f'ROE {round(roe,1) if roe else None}% < {min_roe}% ({tier})', tier

    # ── Rule 3: D/E (exempt financials) ──────────────────────
    if not is_financial:
        de = row.get('Debt to Equity')
        if de is not None and not pd.isna(de) and de > max_de:
            return False, f'D/E {de} > {max_de} ({tier})', tier

    # ── Rule 4: Promoter (exempt MNC + institutional) ─────────
    if not is_mnc and not is_institutional:
        promo = row.get('Promoter Holding')
        if promo is None or pd.isna(promo) or promo < min_promo:
            return False, f'Promoter {promo}% < {min_promo}% ({tier})', tier

    # ── Rule 5: Revenue ───────────────────────────────────────
    rev = row.get('TTM Revenue')
    if rev is None or pd.isna(rev) or rev < min_rev:
        return False, f'Revenue {round(rev,0) if rev else None}Cr < {min_rev}Cr ({tier})', tier

    # ── Rule 6: CF (exempt financials) ───────────────────────
    if not is_financial:
        if 'Consumer Durables' in sector:
            pq = row.get('Profit Positive Q') or 0
            tq = row.get('Total Quarters') or 1
            if pd.isna(pq): pq = 0
            if (pq / tq) < 0.8:
                return False, f'Inconsistent profits: {pq}/{tq} quarters', tier
        else:
            cf_pos   = row.get('CF Positive Years') or 0
            cf_total = row.get('CF Total Years') or 1
            if pd.isna(cf_pos): cf_pos = 0
            if (cf_pos / cf_total) < 0.60:
                return False, f'Poor CF: {cf_pos}/{cf_total} yrs', tier

    # ── Rule 7: Promoter not selling > 5% in 4Q ──────────────
    if not is_mnc and not is_institutional:
        p_change = row.get('Promoter Change 4Q')
        if p_change is not None and not pd.isna(p_change):
            if p_change < -5:
                return False, f'Promoter selling: {p_change}% in 4Q', tier

    # ── Rule 8: Profit positive ≥ 80% quarters ───────────────
    pq = row.get('Profit Positive Q') or 0
    tq = row.get('Total Quarters') or 1
    if pd.isna(pq): pq = 0
    if tq > 0 and (pq / tq) < 0.8:
        return False, f'Inconsistent profits: {pq}/{tq} quarters', tier

    return True, None, tier

# ── APPLY FILTER ─────────────────────────────────────────────
results = []
for _, row in fund_df.iterrows():
    passed, reason, tier = passes_tiered_filter(row)
    results.append({
        'Symbol'       : row['Symbol'],
        'Sector'       : row.get('Sector'),
        'Market_Cap_Cr': row.get('Market_Cap_Cr'),
        'Cap_Tier'     : tier,
        'Passed'       : passed,
        'Fail_Reason'  : reason,
    })

results_df     = pd.DataFrame(results)
passed_quality = results_df[results_df['Passed'] == True]
failed_quality = results_df[results_df['Passed'] == False]

# Save
passed_quality[['Symbol']].to_csv('../data/quality_passed.csv', index=False)
results_df.to_csv('../data/quality_filter_results.csv', index=False)

print(f"\n✅ Tiered quality filter complete")
print(f"   Total input : {len(results_df)}")
print(f"   Passed      : {len(passed_quality)}")
print(f"   Failed      : {len(failed_quality)}")

print(f"\nPassed by tier:")
print(passed_quality['Cap_Tier'].value_counts().to_string())

print(f"\nTop fail reasons:")
print(failed_quality['Fail_Reason']
      .str.split('<').str[0].str.strip()
      .value_counts().head(10).to_string())

print(f"\nPassed by sector (top 15):")
print(passed_quality['Sector'].value_counts().head(15).to_string())



Total stocks for quality filter: 2142

✅ Tiered quality filter complete
   Total input : 2142
   Passed      : 752
   Failed      : 1390

Passed by tier:
Cap_Tier
Large Cap     214
Mid Cap       210
Mini Large    201
Small Cap     127

Top fail reasons:
Fail_Reason
TTM Profit negative or zero    561
ROE nan%                        29
Promoter nan%                   20
ROE 5.6%                        14
Poor CF: 7.0/12.0 yrs           13
ROE 5.8%                        12
ROE 3.0%                        12
ROE 6.8%                        11
ROE 5.7%                        10
ROE 6.6%                        10

Passed by sector (top 15):
Sector
Capital Goods                     130
Financial Services                 91
Healthcare                         68
Information Technology             60
Automobile and Auto Components     59
Chemicals                          57
Fast Moving Consumer Goods         54
Consumer Durables                  40
Consumer Services                  26
Constru